<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformando Datos: Feature Engineering
## Importancia de Variables con Modelos de Árboles

**Objetivo:** Utilizar la métrica de importancia de características de Random Forest en Python para ranquear el poder explicativo de cada variable, enfocando los recursos de ingeniería en los datos más relevantes.

¡Hola! Hoy vamos a descubrir el superpoder de saber exactamente qué datos mueven la aguja en tus modelos. Es frustrante trabajar con cientos de columnas sin saber cuáles realmente aportan valor, pero hoy eso cambia radicalmente para que seas mucho más eficiente en tu trabajo.

### ¿Por qué es importante el Feature Importance?
Imagina que estás intentando predecir si una película será un éxito. Tienes datos sobre el presupuesto, el director, el color de ojos del catering y el clima el día del estreno. Si le das todo esto a un modelo, el ruido de los datos irrelevantes puede confundirlo.

Aquí es donde entra **Random Forest**. Este modelo no solo predice, sino que actúa como un juez que observa cuántas veces se usó cada variable para tomar decisiones importantes en sus miles de árboles internos. Esta capacidad técnica se llama *Feature Importance* y nos permite separar la señal del ruido absoluto.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# 1. Simulación de datos: 50 variables sobre comportamiento de compra
np.random.seed(42)
n_samples = 1000
n_features = 50

# Generamos datos aleatorios
data = np.random.rand(n_samples, n_features)
columns = [f'sensor_{i}' for i in range(n_features)]
df = pd.DataFrame(data, columns=columns)

# Renombramos variables clave para nuestro ejemplo
df['historial_compras'] = df['sensor_0'] * 5  # Variable muy importante
df['color_interfaz_web'] = df['sensor_1']     # Variable irrelevante

# Creamos el target (1 si compra, 0 si no) influenciado fuertemente por el historial
target = (df['historial_compras'] + np.random.normal(0, 0.5, n_samples) > 2.5).astype(int)

# 2. Entrenamiento del modelo
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(df, target)

# 3. Extracción de importancia de variables
importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]  # Top 10

# 4. Visualización
plt.figure(figsize=(10, 6))
plt.title('Importancia de Variables (Top 10)')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [df.columns[i] for i in indices])
plt.xlabel('Importancia Relativa')
plt.show()

print('¡Vòila! Los datos nos muestran que el historial de compras pesa muchísimo más que la estética del sitio.')

### Profundización Técnica: Impureza de Gini

Random Forest utiliza internamente algo llamado **impureza de Gini** o ganancia de información. Cada vez que un árbol decide dividir los datos usando una variable, lo hace para que los grupos resultantes sean más puros o decididos.

Si una variable como el salario permite separar perfectamente a quienes pagan su deuda de quienes no, esa variable reduce mucho la impureza. El modelo suma todas esas reducciones a lo largo de todos los árboles y así es como calcula ese ranking de importancia que acabamos de ver.

### Caso de Uso: La eficiencia de Javier

Pensemos en Javier, un analista en una empresa de logística que lucha con miles de sensores de temperatura. Javier perdía semanas limpiando datos de todos los sensores por igual hasta que usó la importancia de variables de Random Forest.

Descubrió que solo tres sensores en la puerta principal explicaban el noventa por ciento de las variaciones de carga. Al enfocarse solo en esos tres, redujo su tiempo de procesamiento a la mitad y mejoró la precisión de sus alertas de mantenimiento.

### La Trampa de la Alta Cardinalidad

Es vital entender que no todo es perfecto. Un error común es que la importancia basada en impureza suele favorecer injustamente a las variables numéricas con muchísimos valores únicos (alta cardinalidad).

Si tienes un **ID de cliente**, el modelo podría decir que es importantísimo porque ayuda a memorizar los datos, pero no sirve para predecir el futuro. Por eso, recurrimos a la técnica **Permutation Importance**.

In [ ]:
from sklearn.inspection import permutation_importance

# Agregamos una variable de alta cardinalidad (ID de usuario) que es ruido puro
df['user_id'] = np.arange(n_samples)

# Reentrenamos
rf.fit(df, target)

# Importancia estándar (Gini)
gini_importance = rf.feature_importances_

# Importancia por Permutación
perm_importance = permutation_importance(rf, df, target, n_repeats=10, random_state=42)

print('Comparativa de Importancia:')
print(f'Importancia Gini de user_id: {gini_importance[df.columns.get_loc(\'user_id\')]:.4f}')
print(f'Importancia Permutación de user_id: {perm_importance.importances_mean[df.columns.get_loc(\'user_id\')]:.4f}')
print()
print('Nota cómo la permutación reduce a cero la importancia del ID ruidoso.')

### Estrategia de Ingeniería de Características

Si ya identificaste que tu variable de ingresos tiene una importancia altísima pero tiene muchos valores nulos, ahí es donde debes gastar tus balas. No pierdas tiempo creando interacciones complejas con variables que el ranking situó al final.

La eficiencia en Feature Engineering consiste en dedicar el **ochenta por ciento de tu tiempo** a mejorar la calidad y el contexto de ese top 5 de variables que el modelo ya nos marcó como fundamentales.

In [ ]:
import time

# Modelo con 50 variables
start_time = time.time()
rf.fit(df, target)
duration_full = time.time() - start_time

# Modelo con solo las 2 variables principales
top_features = ['historial_compras', 'sensor_5'] # Supongamos estas dos
start_time = time.time()
rf.fit(df[top_features], target)
duration_reduced = time.time() - start_time

print(f'Tiempo con todas las variables: {duration_full:.4f} seg')
print(f'Tiempo con variables reducidas: {duration_reduced:.4f} seg')
print()
print('Al eliminar distracciones, el camino hacia la predicción es más claro y rápido.')

### Conclusión y Cierre

Al reducir el número de variables basándonos en su importancia, ayudamos al modelo a generalizar mejor y evitamos el **overfitting**. Es como limpiar el parabrisas de un coche en medio de una tormenta; de repente, el camino se vuelve mucho más claro porque el algoritmo deja de intentar encontrar patrones en el ruido estadístico aleatorio.

Es vital que mantengan esa curiosidad analítica activa. No se conformen con el primer número; cuestionen siempre por qué una variable es importante y si tiene sentido lógico en el mundo real.

¡Qué avance hemos logrado hoy! Ahora tienen el criterio para decidir en qué datos invertir sus valiosas horas de ingeniería. Sigan practicando y verán cómo su intuición técnica se vuelve cada vez más aguda. ¡Nos vemos en la siguiente lección!